# 3. Aplicações práticas

**CatBoost aplicado à previsão de churn**

Atividade Final de Aprendizado de Máquina I · FATEC Jundiaí · Ciência de Dados

| Aluna | Professor | Tema |
|:--|:--|:--|
| Mariana Costa de Mello | Mateus Guilherme Fuini | CatBoost |

## Casos reais

| Organização | Uso | Fonte |
|:--|:--|:--|
| Yandex | Previsão do tempo, detecção de fraude, otimização industrial | [Comunicado de lançamento (2017)](https://www.yandex.com/company/press_center/press_releases/2017/0718) |
| CERN, experimento LHCb | Identificação de partículas nos detectores | [Comunicado da Yandex (2017)](https://www.yandex.com/company/press_center/press_releases/2017/0718) |
| Cloudflare | Detecção de bots, com inferência em microssegundos | [Notícia no site do CatBoost (2019)](https://catboost.ai/news/cloudflare-s-protection-against-bots-employs-catboost) e [Blog da Cloudflare (2023)](https://blog.cloudflare.com/how-cloudflare-runs-ml-inference-in-microseconds/) |
| JetBrains | Ordenação das sugestões de autocompletar | [Wikipédia, página do CatBoost](https://en.wikipedia.org/wiki/CatBoost) |

## Onde ele se encaixa

| Setor | Exemplo |
|:--|:--|
| Telecom e assinaturas | Churn |
| Bancos | Crédito e inadimplência |
| Pagamentos | Fraude, com IDs de loja e dispositivo (alta cardinalidade) |
| Varejo | Previsão de demanda e preço |
| Busca | Ranqueamento de resultados |

Ele funciona melhor em **dados tabulares com muitas categóricas**. Para imagem, áudio ou texto livre, redes neurais costumam ser a melhor escolha.

## Outras tarefas com a mesma API

Dois exemplos com dados sintéticos: **regressão** do preço de aluguel com o bairro como categórica e **ranqueamento** de produtos por busca com o `CatBoostRanker`.

In [1]:
%%capture
%pip install catboost

In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, CatBoostRanker, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

rng = np.random.default_rng(42)
n = 3000
imoveis = pd.DataFrame({
    "bairro": rng.choice([f"bairro_{i}" for i in range(40)], n),
    "tipo": rng.choice(["apartamento", "casa", "kitnet"], n),
    "area_m2": rng.uniform(25, 180, n).round(),
})
efeito_bairro = dict(zip(sorted(imoveis["bairro"].unique()), rng.normal(0, 600, 40)))
imoveis["aluguel"] = 800 + 18 * imoveis["area_m2"] + imoveis["bairro"].map(efeito_bairro) + rng.normal(0, 250, n)

X_tr, X_te, y_tr, y_te = train_test_split(imoveis.drop(columns="aluguel"), imoveis["aluguel"], random_state=42)
reg = CatBoostRegressor(iterations=500, verbose=0, cat_features=["bairro", "tipo"]).fit(X_tr, y_tr)
print(f"Regressão: R² = {r2_score(y_te, reg.predict(X_te)):.3f} | MAE = R$ {mean_absolute_error(y_te, reg.predict(X_te)):.0f}")

Regressão: R² = 0.931 | MAE = R$ 217


In [3]:
busca = pd.DataFrame({
    "query_id": np.repeat(np.arange(300), 10),
    "marca": rng.choice(["m1", "m2", "m3"], 3000),
    "preco": rng.uniform(10, 200, 3000),
    "similaridade": rng.random(3000),
})
busca["relevancia"] = pd.qcut(3 * busca["similaridade"] + busca["marca"].eq("m2") - busca["preco"] / 200 + rng.normal(0, 0.3, 3000), 4, labels=False)

treino = busca["query_id"] < 240
pools = [Pool(d[["marca", "preco", "similaridade"]], d["relevancia"], group_id=d["query_id"], cat_features=["marca"])
         for d in (busca[treino], busca[~treino])]
ranker = CatBoostRanker(iterations=300, loss_function="YetiRank", verbose=0).fit(pools[0])
print(f"Ranqueamento: NDCG@5 = {ranker.eval_metrics(pools[1], ['NDCG:top=5'])['NDCG:top=5;type=Base'][-1]:.3f}")

Ranqueamento: NDCG@5 = 0.978


> Nos dois casos só muda a classe (`CatBoostRegressor`, `CatBoostRanker`) e as categóricas entram como texto. Os números são de dados sintéticos e servem só para mostrar a API.

No restante do trabalho o caso estudado é **churn em telecom**: 16 de 19 variáveis categóricas, classes desbalanceadas e uso direto no negócio.